# Wavelet (WDM) Domain Tutorial

The Wavelet (WDM) representation gives a unified time-frequency view of a LISA-band signal.
Given a length-`N` time series, the WDM coefficients live on a uniform `(Nf, Nt)` grid with
`Nf * Nt = N / 2`. The transform is unitary and invertible up to filter-edge effects.

This tutorial builds intuition by:
1. Computing the WDM of a pure **sine wave**.
2. Adjusting the sine's **initial phase** and watching what changes (and what doesn't).
3. Computing the WDM of an actual **Galactic Binary** waveform from `gbgpu`.

All transforms live in `lisatools.domains`. Backend dispatch happens at construction time via
`force_backend` (`'cpu'` / `'cuda12x'` / `'jax'` / ...); we use the auto-resolved default below.

*Note on shapes*: `TDSignal.arr` has shape `(nchannels, N)` and `WDMSignal.arr` has shape
`(nchannels, Nf, Nt)` — for the single-channel synthetic signals here we index `[0]` to drop the
leading channel axis.

## Setup

In [ ]:
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from lisatools.domains import TDSettings, WDMSettings, TDSignal, WDMSignal
from lisatools.utils.constants import YRSID_SI

## 1. WDM of a sine wave

Start with the simplest possible LISA-band signal: a monochromatic sinusoid at a single
frequency `f0`. Pick a half-year baseline at 10 s sampling and an amplitude in the GB ballpark.

In [ ]:
dt   = 10.0                       # seconds
Tobs = 0.5 * YRSID_SI             # half year
N    = int(round(Tobs / dt))
N    = 1 << (N.bit_length() - 1)  # round to power of two (required by WDM)
t    = np.arange(N) * dt

f0   = 3.0e-3                     # 3 mHz -- mid-LISA band
amp  = 1.0e-21                    # GB-scale amplitude

phi0 = 0.0                        # initial phase (rad)
h_sin = amp * np.sin(2 * np.pi * f0 * t + phi0)

print(f'N = {N:,} samples ({N*dt/YRSID_SI:.3f} yr at dt = {dt} s)')
print(f'f0 = {f0*1e3} mHz   amp = {amp:.1e}')

Wrap the array in a `TDSignal`. `TDSettings` carries the array length + sample rate; `TDSignal`
binds the data to those settings.

In [ ]:
td_set = TDSettings(N=N, dt=dt)
td_sig_sine = TDSignal(h_sin, settings=td_set)
print('TDSettings:', td_set)
print('TDSignal.arr shape:', td_sig_sine.arr.shape, '  (nchannels, N)')

### Build a WDM grid + run the forward transform

A WDM grid is `(Nf, Nt)` with `Nf * Nt = N / 2`. Layer spacing is `layer_df = 1 / (2*Nf*dt)`
in frequency and `layer_dt = N*dt / Nt` in time -- they're conjugate. Pick `Nf = 256` for
decent frequency resolution (~0.2 mHz per layer at this baseline).

In [ ]:
Nf = 256
Nt = N // (2 * Nf)
wdm_set = WDMSettings(Nf=Nf, Nt=Nt, dt=dt)
print(f'WDM grid: Nf={wdm_set.Nf}  Nt={wdm_set.Nt}  '
      f'layer_df={wdm_set.layer_df*1e3:.4f} mHz  '
      f'layer_dt={wdm_set.layer_dt/3600:.2f} hr')

wdm_sine = td_sig_sine.transform(wdm_set)
wdm_arr  = wdm_sine.arr[0]    # drop the (single) channel axis -> (Nf, Nt)
print('WDMSignal.arr shape:', wdm_sine.arr.shape, '  (nchannels, Nf, Nt)')

# Which m-layer holds the carrier?
m_floor = int(np.floor(f0 / wdm_set.layer_df))
print(f'carrier f0 = {f0*1e3} mHz  ->  m_floor = {m_floor} '
      f'(layer center {m_floor * wdm_set.layer_df * 1e3:.3f} mHz)')

### Plot |w_mn|

Energy concentrates in the two layers `m = m_floor` and `m = m_floor + 1` straddling `f0`, and
is roughly constant across all `n` (time) because the signal is stationary.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.pcolormesh(
    np.arange(Nt) * wdm_set.layer_dt / YRSID_SI,
    np.arange(Nf) * wdm_set.layer_df * 1e3,
    np.abs(wdm_arr),
    shading='auto', cmap='magma',
)
ax.set_xlabel('time [yr]')
ax.set_ylabel('frequency [mHz]')
ax.set_title(f'|w_mn|  --  pure sine at f0 = {f0*1e3} mHz')
ax.set_ylim(0, 6)
fig.colorbar(im, ax=ax, label='|w_mn|')
plt.tight_layout()
plt.show()

### Zoom: which `(m, n)` pixels actually carry the energy?

For a monochromatic source the **WDM signature is two layers wide** in frequency (the carrier
straddles `m_floor` and `m_floor+1`) and **uniform across all n** in time. The relative weight
between the two m-layers depends on how close `f0` sits to a layer boundary.

In [ ]:
n_mid = Nt // 2
print('Coefficients near the carrier at n = Nt/2:')
for m in range(m_floor - 2, m_floor + 4):
    f_layer_mhz = m * wdm_set.layer_df * 1e3
    print(f'  m = {m:3d}  (f = {f_layer_mhz:.3f} mHz)  |w_mn| = {abs(wdm_arr[m, n_mid]):.3e}')

## 2. Effect of the initial phase `phi0`

What happens if we slide the initial phase of the sine wave? Re-run the WDM transform for
several values of `phi0` and compare the magnitudes -- the magnitudes are essentially invariant
to a constant phase shift; the WDM coefficient sign / complex phase per pixel tracks `phi0`.

In [ ]:
phi0_values = [0.0, np.pi/4, np.pi/2, np.pi, 3*np.pi/2]
labels      = ['0', 'pi/4', 'pi/2', 'pi', '3pi/2']

fig, axes = plt.subplots(1, len(phi0_values), figsize=(15, 3.2), sharey=True)
for ax, phi0_val, lab in zip(axes, phi0_values, labels):
    h         = amp * np.sin(2 * np.pi * f0 * t + phi0_val)
    tdsig     = TDSignal(h, settings=td_set)
    wdm_p     = tdsig.transform(wdm_set).arr[0]
    im = ax.pcolormesh(
        np.arange(Nt) * wdm_set.layer_dt / YRSID_SI,
        np.arange(Nf) * wdm_set.layer_df * 1e3,
        np.abs(wdm_p), shading='auto', cmap='magma',
    )
    ax.set_title(f'phi0 = {lab}')
    ax.set_xlabel('time [yr]')
    ax.set_ylim(0, 6)
axes[0].set_ylabel('frequency [mHz]')
plt.tight_layout()
plt.show()

The magnitude maps look **identical**. The phase information lives in the *signed* WDM
coefficients (`w_mn` is real here, not complex -- the WDM transform implemented in lisatools is
a real-valued basis); the `|w_mn|` plot is invariant. Confirm by inspecting the raw signs at the
carrier layer:

In [ ]:
n_mid = Nt // 2
print(f'Raw w_mn at (m_floor={m_floor}, n=Nt/2)  --  carrier pixel\n')
print('  phi0          w_mn (signed)')
for phi0_val, lab in zip(phi0_values, labels):
    h     = amp * np.sin(2 * np.pi * f0 * t + phi0_val)
    wdm_p = TDSignal(h, settings=td_set).transform(wdm_set).arr[0]
    w     = wdm_p[m_floor, n_mid]
    print(f'  {lab:6s}    {w:+.4e}')

The magnitudes match; the sign-and-magnitude pattern tracks `phi0` modulo the WDM basis
convention. This is the central reason chunked-heterodyne / signal-het kernels work entirely in
the WDM domain: phase information is preserved pixel-by-pixel.

## 3. WDM of a Galactic Binary waveform

Now do the same with a real Galactic Binary waveform from `gbgpu.GBGPU`. This is the workflow
the LDC / global-fit pipelines use to convert injected GB sources into WDM data.

In [ ]:
from gbgpu.gbgpu import GBGPU

gb = GBGPU()

# GBGPU expects every parameter as a length-(num_bin) array.
amp_gb   = np.array([1.084702251e-22])
f0_gb    = np.array([2.35962078e-3])
fdot_gb  = np.array([1.47197271e-17])
fddot_gb = np.array([0.0])
phi0_gb  = np.array([4.91128699])
iota_gb  = np.array([1.11820901])
psi_gb   = np.array([2.3290324])
lam_gb   = np.array([5.22979888])
beta_gb  = np.array([0.9805742971871619])

# T = N * dt is the dense time-domain duration in seconds.
T_sec = N * dt

gb.run_wave(
    amp_gb, f0_gb, fdot_gb, fddot_gb, phi0_gb, iota_gb, psi_gb, lam_gb, beta_gb,
    N=N, dt=dt, T=T_sec, tdi_channel_setup='AE',
)
# Default 'AE' channel setup -> gb.A, gb.E populated in FD on the dense grid.
# Inverse-FFT one channel to get a TD signal for the WDM transform.
Af = np.asarray(gb.A[0])
At = np.fft.irfft(Af, n=N).real
print('|At| peak  =', float(np.abs(At).max()))
print('TD samples =', At.shape)

Wrap the GB time series in a `TDSignal` and run the WDM transform:

In [ ]:
td_gb      = TDSignal(At, settings=td_set)
wdm_gb     = td_gb.transform(wdm_set)
wdm_arr_gb = wdm_gb.arr[0]

m_floor_gb = int(np.floor(f0_gb[0] / wdm_set.layer_df))
print(f'GB carrier f0 = {f0_gb[0]*1e3:.4f} mHz  ->  m_floor = {m_floor_gb}')
print('WDMSignal.arr shape:', wdm_gb.arr.shape, '  (Nf, Nt) =', wdm_arr_gb.shape)

Plot the GB's WDM signature. Two features to look for:
- A bright band hugging `m_floor` -- the carrier.
- A slow drift across `n` (time) because GBs evolve under `fdot`; on the half-year baseline
  this is just visible as a slight slope.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))

ax = axes[0]
im = ax.pcolormesh(
    np.arange(Nt) * wdm_set.layer_dt / YRSID_SI,
    np.arange(Nf) * wdm_set.layer_df * 1e3,
    np.abs(wdm_arr_gb),
    shading='auto', cmap='magma',
)
ax.set_xlabel('time [yr]')
ax.set_ylabel('frequency [mHz]')
ax.set_title('|w_mn|  --  Galactic Binary')
ax.set_ylim(max(0, f0_gb[0]*1e3 - 1), f0_gb[0]*1e3 + 1)
fig.colorbar(im, ax=ax, label='|w_mn|')

ax = axes[1]
ax.plot(np.arange(Nf) * wdm_set.layer_df * 1e3,
        np.linalg.norm(wdm_arr_gb, axis=1))
ax.axvline(f0_gb[0] * 1e3, color='r', ls='--', alpha=0.6, label=f'f0 = {f0_gb[0]*1e3:.3f} mHz')
ax.set_xlabel('frequency [mHz]')
ax.set_ylabel('sqrt(sum_n |w_mn|^2)')
ax.set_title('energy per m-layer')
ax.set_xlim(f0_gb[0]*1e3 - 0.5, f0_gb[0]*1e3 + 0.5)
ax.legend()

plt.tight_layout()
plt.show()

## Recap & what's next

- `TDSettings(N, dt)` + `TDSignal(arr, settings=...)` carry a time-domain array.
- `WDMSettings(Nf, Nt, dt)` describes the wavelet grid; `WDMSignal` carries the coefficient array
  via `.arr` (shape `(nchannels, Nf, Nt)`).
- The forward transform is `td_sig.transform(wdm_set)`; the inverse is `wdm_sig.transform(td_set)`.
- For monochromatic sources, the WDM signature concentrates in two adjacent `m` layers and is
  uniform in `n`. Changing the initial phase only re-distributes the sign / phase per pixel --
  magnitudes are preserved.
- For a Galactic Binary, the carrier band sits at `m_floor = floor(f0 / layer_df)`; the slight
  drift across `n` reflects `fdot` evolution.

The chunked-heterodyne (`gbgpu/examples/chunked_het_tutorial.ipynb`) and signal-het
(`gbgpu/examples/signal_het_tutorial.ipynb`) tutorials show how these representations power the
fast GB likelihoods that drive the global fit pipeline.